# What the agent actually returns

The sibling notebook, [`Strands_Evals_Evaluator.ipynb`](Strands_Evals_Evaluator.ipynb), builds its
predictions by hand: it never runs an agent, so nothing in it shows what an agent's response looks like.
[`Strands_Evals_FCC_Live_Agent.ipynb`](Strands_Evals_FCC_Live_Agent.ipynb) does run one, but it needs
Bedrock credentials, so its outputs are empty until you supply them.

This notebook fills the gap. One extraction, end to end, one step per cell:

1. the tool specification Strands derives from your Pydantic model — what the model is *actually* asked
2. the raw tool-use payload the model returns — what "the agent responded" really means
3. how Strands turns that payload into a validated object
4. what stickler scores it as, field by field
5. exactly where the points went, and why each field got the comparator and threshold it did

**No live inference and no credentials.** A recorded Bedrock exchange replays through a stub model
provider, so the real `Agent` structured-output path executes and the output is identical on every run.
Scoring uses `stickler.evaluate()` directly, so no Strands Evals install is needed either.


## Install

```bash
pip install stickler-eval strands-agents
```

Both are on PyPI; nothing here is pinned to a fork. Unlike the other two notebooks, this one needs no
`strands-agents-evals` install, because it scores with stickler directly rather than through the
evaluator.


In [1]:
import datetime
import json
from typing import Any, Optional

from pydantic import BaseModel
from strands import Agent
from strands.models import Model
from strands.tools.structured_output import convert_pydantic_to_tool_spec

import stickler

print(f"stickler {stickler.__version__}")

stickler 1.0.0


## 1. The model, and the tool spec Strands builds from it

`structured_output_model=Invoice` does not put your model in the prompt. Strands converts it into a
*tool* and offers that tool to the model; the extraction is the model calling it. The schema below is
therefore the real interface the model is programmed against — anything ambiguous in your field names
is ambiguous to the model too.


In [2]:
class LineItem(BaseModel):
    sku: str
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: datetime.date
    total_amount: float
    po_number: Optional[str] = None
    line_items: list[LineItem] = []


spec = convert_pydantic_to_tool_spec(Invoice)
schema = spec["inputSchema"]["json"]

print(f"tool name : {spec['name']}")
print(f"required  : {schema['required']}")
print("properties:")
for name, prop in schema["properties"].items():
    kind = prop.get("type") or [a.get("type") for a in prop.get("anyOf", [])]
    print(f"  {name:16} {kind}")

tool name : Invoice
required  : ['invoice_id', 'vendor_name', 'invoice_date', 'total_amount']
properties:
  invoice_id       string
  vendor_name      string
  invoice_date     string
  total_amount     number
  po_number        ['string', 'null']
  line_items       ['array', 'null']


Three things to read off that spec:

- **The tool is named after the class.** `Invoice`, not `structured_output`. When you inspect what tools
  an agent was offered, that is the name you will see.
- **`required` holds exactly the fields with no default.** `po_number` and `line_items` are absent
  because both have one, so the model is *permitted* to omit them. It will.
- **`invoice_date` is a `string`.** JSON has no date type, so the date crosses the wire as text and is
  parsed on arrival. That is why a date comparator matters later: `2024-03-15` and `March 15, 2024` are
  the same date and different strings.


## 2. The document and the label

`DOCUMENT` is what the agent sees. `GROUND_TRUTH` is the human label to score against — the
`expected_output` you would put on a `Case`.


In [3]:
DOCUMENT = """
INVOICE  #INV-2024-0042
Acme Corporation                    Date: 2024-03-15
PO: PO-88231

  2 x Wireless Mouse        (WM-100)  @  $29.99
  5 x USB-C Cable 1m        (UC-050)  @  $12.99
  1 x Mechanical Keyboard   (KB-200)  @  $89.99

Total: $1,247.50            Terms: Net 30
"""

GROUND_TRUTH = Invoice(
    invoice_id="INV-2024-0042",
    vendor_name="Acme Corporation",
    invoice_date=datetime.date(2024, 3, 15),
    total_amount=1247.50,
    po_number="PO-88231",
    line_items=[
        LineItem(sku="WM-100", description="Wireless Mouse", quantity=2, unit_price=29.99),
        LineItem(sku="UC-050", description="USB-C Cable 1m", quantity=5, unit_price=12.99),
        LineItem(sku="KB-200", description="Mechanical Keyboard", quantity=1, unit_price=89.99),
    ],
)

print(f"{len(GROUND_TRUTH.line_items)} line items labelled")

3 line items labelled


## 3. The recorded exchange

`fixtures/strands_offline_agent.json` holds one agent turn: the tool the model chose, the `toolUseId`,
and the JSON it passed as input. That JSON *is* the agent's answer.

A real provider streams that JSON in fragments, so the fixture keeps the fragment boundaries. Replaying
them rather than handing over the whole string exercises the same accumulate-then-parse path a live run
takes — which is where malformed-JSON failures actually surface.


In [4]:
with open("fixtures/strands_offline_agent.json") as fh:
    FIXTURE = json.load(fh)

print(f"recorded from : {FIXTURE['provenance']['model_id']}")
print(f"provenance    : {FIXTURE['provenance']['note']}\n")

turn = FIXTURE["turns"][0]["toolUse"]
print(f"tool called   : {turn['name']}")
print(f"toolUseId     : {turn['toolUseId']}")
print(f"streamed in   : {len(turn['inputChunks'])} fragments\n")
print("first three fragments, exactly as they arrive:")
for chunk in turn["inputChunks"][:3]:
    print(f"  {chunk!r}")

recorded from : us.anthropic.claude-haiku-4-5-20251001-v1:0
provenance    : Hand-authored to the Bedrock converse-stream wire shape so this notebook runs with no AWS account. Regenerate from a live agent with fixtures/record_agent_exchange.py.

tool called   : Invoice
toolUseId     : tooluse_offline_demo_0001
streamed in   : 11 fragments

first three fragments, exactly as they arrive:
  '{"invoice_id": "INV-2024-0042", "vend'
  'or_name": "Acme Corp", "invoice_date"'
  ': "2024-03-15", "total_amount": 1247.'


The fragments split mid-token — `"vend"` / `"or_name"`. Nothing is parseable until the last one lands,
which is why the event loop buffers the whole string before validating it, and why a truncated response
fails as a JSON error rather than as a missing field.


## 4. The stub model provider

A Strands model provider implements `stream`: an async generator of `StreamEvent` dicts. `ReplayModel`
emits the recorded events instead of calling a service. Everything downstream — the event loop, the
structured-output tool, Pydantic validation — is the real code path.

Two details worth copying if you write your own:

- The tool-use block opens with `{"contentBlockStart": {"start": {"toolUse": {...}}}}`. The inner
  `toolUse` key is required. Without it the block is read as text, the tool is never invoked, and the
  run fails with `StructuredOutputException` rather than anything that points at the cause.
- `stream` raises when the agent asks for a turn the fixture does not have, instead of repeating the
  last one. A replay that silently loops looks like a passing test while proving nothing.


In [5]:
class ReplayModel(Model):
    """Replays a recorded exchange. No network, no credentials, same code path."""

    def __init__(self, turns: list[dict[str, Any]]) -> None:
        self._turns = turns
        self._calls = 0
        self.tool_specs_seen: list[str] = []

    def get_config(self) -> dict[str, Any]:
        return {"replay": True, "turns": len(self._turns)}

    def update_config(self, **kwargs: Any) -> None:
        pass

    async def structured_output(self, output_model, prompt, system_prompt=None, **kwargs):
        # Reached only by the older agent.structured_output() API, not by this notebook.
        yield {"output": output_model.model_validate(self._turns[0]["toolUse"]["input"])}

    async def stream(self, messages, tool_specs=None, system_prompt=None, **kwargs):
        self.tool_specs_seen = [t.get("name") for t in (tool_specs or [])]
        if self._calls >= len(self._turns):
            raise AssertionError(
                f"replay exhausted: the agent asked for turn {self._calls + 1}, "
                f"the fixture records {len(self._turns)}"
            )
        turn = self._turns[self._calls]["toolUse"]
        self._calls += 1

        yield {"messageStart": {"role": "assistant"}}
        yield {"contentBlockStart": {"start": {"toolUse": {
            "name": turn["name"], "toolUseId": turn["toolUseId"]}}}}
        for chunk in turn["inputChunks"]:
            yield {"contentBlockDelta": {"delta": {"toolUse": {"input": chunk}}}}
        yield {"contentBlockStop": {}}
        yield {"messageStop": {"stopReason": "tool_use"}}
        yield {"metadata": {"usage": turn["usage"], "metrics": {"latencyMs": 0}}}


print("ReplayModel ready")

ReplayModel ready


## 5. Run the agent

Ordinary Strands. The only difference from a live run is which `model=` is passed.


In [6]:
model = ReplayModel(FIXTURE["turns"])
agent = Agent(model=model, system_prompt="You extract invoice data.", callback_handler=None)

result = await agent.invoke_async(
    f"Extract the invoice fields.\n\nDOCUMENT:\n{DOCUMENT}",
    structured_output_model=Invoice,
)

print(f"stop_reason      : {result.stop_reason}")
print(f"tools offered    : {model.tool_specs_seen}")
print(f"model turns used : {model._calls}")
print(f"structured_output: {type(result.structured_output).__name__}\n")
print(result.structured_output.model_dump_json(indent=2))

stop_reason      : tool_use
tools offered    : ['Invoice']
model turns used : 1
structured_output: Invoice

{
  "invoice_id": "INV-2024-0042",
  "vendor_name": "Acme Corp",
  "invoice_date": "2024-03-15",
  "total_amount": 1247.5,
  "po_number": null,
  "line_items": [
    {
      "sku": "UC-050",
      "description": "USB-C Cable 1 m",
      "quantity": 5,
      "unit_price": 12.99
    },
    {
      "sku": "WM-100",
      "description": "Wireless Mouse",
      "quantity": 2,
      "unit_price": 29.99
    },
    {
      "sku": "KB-200",
      "description": "Mechanical Keyboard",
      "quantity": 1,
      "unit_price": 8.99
    }
  ]
}


`stop_reason` is `tool_use`, not `end_turn`: the run ends when the structured-output tool validates, so
one model turn is all a clean extraction costs. `tools offered` confirms the model saw exactly one tool,
named for the class.

`structured_output` is a real `Invoice`. Strands buffered the fragments, parsed them, and validated
through Pydantic — so `invoice_date` is a `datetime.date` here even though it arrived as the string
`"2024-03-15"`, and `po_number` is `None` rather than missing.

Now read the values against the document. The vendor came back abbreviated, `po_number` was dropped, the
keyboard's price is `8.99` instead of `89.99`, and the line items are in a different order than the
label. That is a realistic extraction, and a single pass/fail would tell you none of it.


## 6. Score it

The whole integration is one call. No `StructuredModel`, no comparators, no thresholds, no schema —
`stickler.evaluate` infers a comparator and threshold per field from the Pydantic model.


In [7]:
prediction = result.structured_output
evaluation = stickler.evaluate(GROUND_TRUTH, prediction)

print(f"overall score : {evaluation.overall_score:.3f}")
print(f"matched       : {evaluation.matched}")
print(f"precision     : {evaluation.precision:.3f}")
print(f"recall        : {evaluation.recall:.3f}")
print(f"f1            : {evaluation.f1:.3f}\n")

print(f"{'field':16} {'score':>6}")
print("-" * 24)
for field, score in evaluation.field_scores.items():
    print(f"{field:16} {score:>6.3f}")

overall score : 0.652
matched       : False
precision     : 0.857
recall        : 0.857
f1            : 0.857

field             score
------------------------
invoice_id        1.000
vendor_name       0.000
invoice_date      1.000
total_amount      1.000
po_number         0.000
line_items        0.914


`overall_score` is the mean of those six field scores at default uniform weights, and `matched` is that
score against the `0.7` default threshold — so this document fails.

The breakdown says four things the single number cannot:

- **`invoice_id`, `invoice_date` and `total_amount` score 1.000.** The fields that usually matter most
  were extracted exactly.
- **`line_items` scores 0.914 despite the reordering.** Order independence is the point: stickler pairs
  elements by best match before scoring, so the wrong price is what costs, not the shuffle. Had order
  mattered, all three would have been penalised and the field would look catastrophically wrong.
- **`po_number` scores 0.000.** The value was on the document and the model left it out — and note the
  tool spec permitted that, because the field has a default.
- **`vendor_name` also scores 0.000**, even though `"Acme Corp"` is plainly close to
  `"Acme Corporation"`. Field scores are threshold-gated: below the threshold the score is zeroed, not
  reduced. So a near miss and a total miss read identically here. The next cell recovers what was
  actually lost.


## 7. Where the points went

`field_scores` says a field failed. `non_matches` says *how* — the category, both values, the raw
similarity, and the threshold it fell short of. This is the cell to reach for when a score surprises you.


In [8]:
for nm in evaluation.non_matches:
    print(f"{nm['field_path']:26} {nm['non_match_type'].value}")
    print(f"  ground truth : {nm['ground_truth_value']!r}")
    print(f"  prediction   : {nm['prediction_value']!r}")
    if "similarity_score" in nm:
        print(f"  similarity   : {nm['similarity_score']:.3f}")
    reason = nm.get("reason") or nm.get("details", {}).get("reason")
    print(f"  reason       : {reason}\n")

overall = evaluation.confusion_matrix["overall"]
print("document-level counts")
print(f"  tp={overall['tp']}  fn={overall['fn']}  fa={overall['fa']}  fd={overall['fd']}  tn={overall['tn']}")

nested = evaluation.confusion_matrix["fields"]["line_items"]["fields"]
print(f"\n{'line_items leaf':20} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 54)
for leaf, cm in nested.items():
    o, d = cm["overall"], cm["overall"]["derived"]
    print(f"{leaf:20} {o['tp']:>3} {o['fn']:>3} {o['fa']:>3} {o['fd']:>3}  "
          f"{d['cm_precision']:>5.2f} {d['cm_recall']:>5.2f} {d['cm_f1']:>5.2f}")

vendor_name                false_discovery
  ground truth : 'Acme Corporation'
  prediction   : 'Acme Corp'
  similarity   : 0.562
  reason       : below threshold (0.562 < 0.85)

po_number                  false_negative
  ground truth : 'PO-88231'
  prediction   : None
  reason       : unmatched ground truth

line_items[2].unit_price   false_discovery
  ground truth : 89.99
  prediction   : 8.99
  similarity   : 0.000
  reason       : field mismatch

document-level counts
  tp=6  fn=1  fa=0  fd=1  tn=0

line_items leaf       tp  fn  fa  fd   prec   rec    f1
------------------------------------------------------
sku                    3   0   0   0   1.00  1.00  1.00
description            3   0   0   0   1.00  1.00  1.00
quantity               3   0   0   0   1.00  1.00  1.00
unit_price             2   0   0   1   0.67  1.00  0.80


Now the three failures are distinguishable, and they need three different fixes:

- **`vendor_name` is `false_discovery`** — found, but scored `0.562` against a `0.85` threshold. Nothing
  is wrong with the extraction; the threshold is wrong for a field where vendors legitimately abbreviate.
  Lower it, or use a comparator that normalises company suffixes.
- **`po_number` is `false_negative`** — nothing was extracted. That is a prompt or schema problem, not a
  comparator one.
- **`line_items[2].unit_price` is `false_discovery`** — `89.99` read as `8.99`, similarity `0.0`. A
  dropped digit is an OCR or model problem, and the one failure here that is actually a data error.

Two things about the counts underneath. First, `description` scores `tp=3`: `"USB-C Cable 1 m"` matched
`"USB-C Cable 1m"` because the inferred fuzzy comparator tolerates the space. Whitespace drift is not a
finding, and stickler does not report it as one.

Second, and the more important one: the **document-level row shows `fd=1`, not `fd=2`**. The wrong price
does not appear there. Nested counts sit behind their parent — the line-item pair matched, so the
document row records it as a true positive and the price error is only visible in the leaf table, where
`unit_price` reads `tp=2 fd=1`, precision `0.67`. Those leaf rows also cover only pairs whose parent
scored at or above `match_threshold`; an element too different to pair emits no leaf rows at all and
lands as `fd` on `line_items`. A per-field dashboard built from document-level counts alone will
under-report nested errors.


## 8. Why each field scored that way

Nothing above was configured, so every comparator and threshold was inferred. `explain()` reports what
was chosen and on what basis, which is what makes an inferred score defensible rather than magic.


In [9]:
print(f"{'field':22} {'comparator':42} {'thr':>5}  basis")
print("-" * 78)
for field, cfg in evaluation.explain().items():
    print(f"{field:22} {cfg['comparator']:42} {cfg['threshold']:>5}  {cfg['source']}")

field                  comparator                                   thr  basis
------------------------------------------------------------------------------
invoice_id             ExactComparator                              1.0  name-token
vendor_name            LevenshteinComparator                       0.85  name-token
invoice_date           DateComparator                              0.95  name-token
total_amount           NumericComparator                           0.95  name-token
po_number              LevenshteinComparator                        0.7  type
line_items             Hungarian (per-element StructuredModel)      0.7  type
line_items.sku         ExactComparator                              1.0  name-token
line_items.description FuzzyComparator                              0.6  name-token
line_items.quantity    NumericComparator                            1.0  name-token
line_items.unit_price  NumericComparator                           0.95  name-token


`basis` is how the choice was made, and it is mostly **`name-token`** — inferred from the field *name*,
not its annotation. `invoice_date` got a `DateComparator` at `0.95` because the name contains `date`;
`total_amount` a `NumericComparator`; `invoice_id` and `sku` an `ExactComparator` at `1.0`, because an
identifier that is nearly right is wrong. `po_number` and `line_items` fall back to **`type`**, the
annotation, at the `0.7` default.

That is also the explanation for section 7. `vendor_name` matched a name token for company names, which
carries a strict `0.85` — appropriate for catching a wrong vendor, too strict for an abbreviated one. The
table is where you find that out before shipping a threshold you did not choose.

The nested `line_items.*` rows appear because `explain()` recurses into element models. `line_items`
itself reports `Hungarian (per-element StructuredModel)` rather than a comparator, which is accurate: a
list of models is paired element-by-element and compared recursively, so it has no single comparator.

If any row is wrong for your data, that is the signal to stop inferring and declare the field
explicitly. See
[Choosing a configuration path](https://awslabs.github.io/stickler/Getting-Started/choosing-a-configuration-path/).


## Re-recording the fixture

The fixture ships hand-authored to the Bedrock wire shape, so this notebook runs for everyone with no AWS
account. To replace it with a genuine capture from your own agent:

```bash
cd fixtures && AWS_PROFILE=<profile> python record_agent_exchange.py
```

That script wraps a real `BedrockModel`, runs this same prompt and schema, and records the tool-use turn
with the fragment boundaries and token usage Bedrock actually emitted. It reports a missing install,
absent credentials, a failed call and a model that never invoked the tool as four distinct errors, and
leaves the fixture untouched on any of them — a real failure is never written in as if it were an agent
response. Re-run this notebook afterwards and every number above is your agent's.

The scores will move, and that is the point: the fixture is the agent's answer, and a different agent
gives a different answer. What does not move is the code path.
